In [ ]:
CSV_PATH    = "jobs_for_embeddings.csv"              # path to your jobs CSV file
ES_URL      = "https://localhost:9200" # Elasticsearch URL
ES_USER     = "elastic"
ES_PASS     = "Dv0*PBZNeZLfkoaPY36L"
SLEEP_SEC   = 0.0                     # optional delay between rows (0 = max speed)
NUM_WORKERS = 8                       # threads for parallel embedding & indexing
BATCH_SIZE  = 100                     # rows per bulk-index batch


## 1 — Imports

In [ ]:
import time
import json
import hashlib

import numpy as np
import pandas as pd


In [ ]:
from tqdm.notebook import tqdm 

from elasticsearch import Elasticsearch

In [ ]:
from sentence_transformers import SentenceTransformer 

In [ ]:
print("Loading JobBERT — this may take a minute on first run...")
jobbert = SentenceTransformer("TechWolf/JobBERT-v2")
DIM = jobbert.get_embedding_dimension()  
EMBEDDING_DIM = DIM * 3                           
print(f"Model ready. Segment dim: {DIM} | Final embedding dim: {EMBEDDING_DIM}")

In [ ]:
es = Elasticsearch(
    ES_URL,
    basic_auth=(ES_USER, ES_PASS),
    verify_certs=False,
    ssl_show_warn=False,
)

In [ ]:


mapping = {
    "mappings": {
        "properties": {
            "job_id":          {"type": "keyword"},
            "normal_job_title":       {"type": "text"},
            "job_description": {"type": "text"},
            "skills":          {"type": "keyword"},
            "embedding": {
                "type":       "dense_vector",
                "dims":       EMBEDDING_DIM,
                "index":      True,
                "similarity": "cosine",
            },
        }
    }
}

try:
    exists = es.indices.exists(index=INDEX_NAME)
except Exception as e:
    print(f"Error checking index existence: {e}")
    if getattr(e, "status_code", None) == 401:
        print("Authentication failed (401). Check ES_URL, ES_USER and ES_PASS.")
    raise

if not exists:
    try:
        resp = es.indices.create(index=INDEX_NAME, body=mapping)
        print(f"Created index '{INDEX_NAME}': {resp}")
    except Exception as e:
        print(f"Error creating index '{INDEX_NAME}': {e}")
        raise
else:
    print(f"Index '{INDEX_NAME}' already exists — skipping creation")

In [ ]:
df = pd.read_csv("companies_jobs.csv")
print(f"Loaded {len(df):,} rows")
INDEX_NAME = "jobs_strat2_actual2"

In [ ]:
import numpy as np

def generate_job_embedding(title: str, description: str, skills: list[str]) -> list[float]:
    """
    Creates 3 separate embeddings (1024 each) and concatenates them into:
    [title | description | skills] = 3072-dim vector
    """

    title = title if title else "unknown"
    description = description if description else ""
    skills_text = " ".join(skills) if skills else "general"

    # IMPORTANT:
    # This keeps SAME structure but avoids extra Python overhead logic

    title_emb = jobbert.encode(title, convert_to_numpy=True)
    desc_emb = jobbert.encode(description, convert_to_numpy=True)
    skills_emb = jobbert.encode(skills_text, convert_to_numpy=True)

    return np.concatenate([title_emb, desc_emb, skills_emb]).tolist()


# Apply cleaning
df["_job_id"] = df["job_post_id"]
df["_title"]   = df["job_title"]
df["_skills"]  = df["skills"].apply(lambda x: [s.strip() for s in str(x).split("|") if s.strip()])
df["_summary"] = df["job_description"]

before = len(df)
df = df[df["_title"] != ""].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no title. Ready to embed: {len(df):,}")

In [ ]:
# trial
for _, row in df.head(3).iterrows():
    emb = generate_job_embedding(row["_title"], row["_summary"], row["_skills"])
    print(f"[OK] '{row['_title']}' → {len(emb)} dims | skills: {row['_skills'][:3]}")

In [ ]:
import numpy as np
import time
import logging
import torch


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("es_index_pipeline")


BATCH_SIZE = 64

indexed_count = 0
errors = []


def generate_job_embedding(title: str, description: str, skills: list[str]) -> list[float]:

    title_emb = jobbert.encode(title if title else "unknown")
    desc_emb = jobbert.encode(description if description else "")
    skills_emb = jobbert.encode(" ".join(skills) if skills else "general")

    return np.concatenate([title_emb, desc_emb, skills_emb]).tolist()


logger.info("Preparing dataset...")



rows = df.to_dict("records")

logger.info(
    f"Dataset ready | total rows = {len(rows):,} | skipped jobs with job_id <= 61,632"
)


jobbert.eval()


for i in range(0, len(rows), BATCH_SIZE):

    batch = rows[i:i + BATCH_SIZE]
    start_time = time.time()

    try:
        logger.info(f"Batch {i} START | size={len(batch)}")

        
        embeddings = []

        with torch.inference_mode():

            for r in batch:

                t0 = time.time()

                emb = generate_job_embedding(
                    r["_title"],
                    r["_summary"],
                    r["_skills"]
                )

                embeddings.append(emb)

                logger.info(
                    f"[EMBEDDED] job_id={r['_job_id']} | "
                    f"title='{r['_title'][:40]}' | "
                    f"time={time.time() - t0:.2f}s | "
                    f"dim={len(emb)}"
                )

        logger.info(f"Batch {i} EMBEDDING DONE in {time.time() - start_time:.2f}s")

      
        index_start = time.time()
        success = 0

        for r, emb in zip(batch, embeddings):

            try:
                es.index(
                    index=INDEX_NAME,
                    id=str(r["_job_id"]),
                    document={
                        "job_id": str(r["_job_id"]),
                        "normal_job_title": r["_title"],
                        "job_description": r["_summary"],
                        "skills": r["_skills"],
                        "embedding": emb
                    }
                )

                success += 1

                logger.info(
                    f"[INDEXED] job_id={r['_job_id']} | "
                    f"title='{r['_title'][:40]}'"
                )

            except Exception as e:
                errors.append({
                    "job_id": r["_job_id"],
                    "error": str(e)
                })

                logger.error(
                    f"[INDEX ERROR] job_id={r['_job_id']} | error={str(e)}"
                )

        indexed_count += success

        logger.info(
            f"Batch {i} INDEX DONE | "
            f"success={success} | "
            f"time={time.time() - index_start:.2f}s"
        )

        logger.info(
            f"Batch {i} TOTAL TIME = {time.time() - start_time:.2f}s"
        )

    except Exception as e:
        logger.exception(f"Batch {i} FAILED: {str(e)}")


logger.info(f"Indexed: {indexed_count:,}")
logger.info(f"Errors : {len(errors):,}")


In [ ]:
if errors:
    err_df = pd.DataFrame(errors)
    display(err_df)
    err_df.to_csv("embedding_errors.csv", index=False)
    print("Saved to embedding_errors.csv")
else:
    print("No errors — all jobs indexed successfully.")

In [ ]:
# Build a query embedding from the first row's title + skills
sample = df.iloc[0]

query_vector = np.concatenate([
    jobbert.encode(sample["_title"]),
    np.zeros(DIM),                                                         # zero-pad description slot
    jobbert.encode(" ".join(sample["_skills"]) if sample["_skills"] else "general"),
]).tolist()

response = es.search(
    index=INDEX_NAME,
    body={
        "knn": {
            "field":          "embedding",
            "query_vector":   query_vector,
            "k":              5,
            "num_candidates": 50,
        },
        "_source": ["job_id", "job_title", "skills"],
    },
)

print(f"Top 5 matches for '{sample['_title']}':")
for hit in response["hits"]["hits"]:
    src = hit["_source"]
    print(f"  [{hit['_score']:.4f}] {src['job_title']}")

In [ ]:

count = es.count(index=INDEX_NAME)["count"]
print(f"Total documents in '{INDEX_NAME}': {count}")